# 🔥 Advanced · MDM — text-to-motion

> 🔥 **Advanced / heavy lab.** Type a sentence, get an animated 3D human motion.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **15–25 min** including downloads. Built on the official **[GuyTevet/motion-diffusion-model](https://github.com/GuyTevet/motion-diffusion-model)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Maps to lesson **A7 (motion diffusion)** — this is the real, pretrained version of the toy DDPM you trained from scratch.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — generate from a pretrained checkpoint | 1× A100 40 GB (or V100 32 GB) to train from scratch |
| **Storage** | ~3 GB (SMPL + GloVe + 1 checkpoint) | HumanML3D ~4 GB + checkpoints ~1 GB; ~20 GB disk |
| **Time** | ~1–2 min / sample | ~1–2 days for ~500k steps |

**Full pipeline (scale-up):** `python -m train.train_mdm --dataset humanml --save_dir save/run --num_steps 500000` (single GPU).

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L

## 1 · Clone + install

In [ ]:
%cd /content
!git clone https://github.com/GuyTevet/motion-diffusion-model.git
%cd motion-diffusion-model
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q "smplx[all]" chumpy blobfile spacy ftfy matplotlib
!python -m spacy download en_core_web_sm

## 2 · Download body files, GloVe, evaluators, and a pretrained HumanML3D model
These helper scripts pull the SMPL files and the `humanml_trans_enc_512` checkpoint.

In [ ]:
!bash prepare/download_smpl_files.sh
!bash prepare/download_glove.sh
!bash prepare/download_t2m_evaluators.sh
!bash prepare/download_pretrained_models.sh

## 3 · Generate motion from YOUR prompt

In [ ]:
!python -m sample.generate \
  --model_path ./save/humanml_trans_enc_512/model000200000.pt \
  --num_repetitions 2 \
  --text_prompt "a person walks forward, stops, and waves with the right hand"

## 4 · Watch the result

In [ ]:
import glob
from IPython.display import Video, display
vids = sorted(glob.glob("save/humanml_trans_enc_512/**/*.mp4", recursive=True))
print(vids[:5])
if vids: display(Video(vids[0], embed=True))

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`.

## How this links to tracks A–D
Generative motion connects out:
- **LM** the same diffusion recipe as image/text generation · **AG** motion priors for agent policies.

## Troubleshooting & next steps
- **chumpy / numpy**: chumpy needs an older NumPy — if import fails, `pip install "numpy<1.24" chumpy`.
- **download scripts fail**: they use `gdown`; rerun, or download the model manually and place it under `save/`.
- **Train your own** instead of using the checkpoint: `python -m train.train_mdm --save_dir save/my_run --dataset humanml` (needs the HumanML3D dataset; multi-hour on a GPU).